# Fair Retail Fish Price Checker for Colombo: Ensemble Learning
**CM4371 Machine Learning & Pattern Recognition, Hands-on Assignment (Ensemble Learning Dashboard)**

## Problem
Fish is the main animal protein for many Sri Lankan households, and consumer prices rose sharply after the 2022 economic crisis.
Fish moves from the **Peliyagoda wholesale market** to retail markets around Colombo (Borella, Nugegoda, Maharagama and others).

**Question:** given this week's wholesale price of a species, what *retail* price should a Colombo shopper expect?

**Data:** weekly fish price reports from the Ministry of Fisheries (Rs/kg), Jan 2019 to Apr 2026, 31 species,
from the `nuuuwan/lk_datasets` collection (dataset 024, *Fisheries Weekly Fish Prices Reports*).

## 0. Setup

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm"], check=False)

In [ ]:
import os, re, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.ensemble import (BaggingRegressor, RandomForestRegressor, AdaBoostRegressor,
                              GradientBoostingRegressor, StackingRegressor, VotingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("Libraries loaded")

## 1. Download the original weekly reports
The Ministry of Fisheries publishes one Excel report per week. The `nuuuwan/lk_fisheries` GitHub repository
archives these original Excel files (`doc.xlsx`, one folder per week).

This cell downloads the repository at a fixed commit, so everyone gets the same 277 reports (Jan 2019 - Apr 2026).
Needs internet access; there is no offline fallback.

In [ ]:
import subprocess, glob

REPO = "https://github.com/nuuuwan/lk_fisheries.git"
SHA = "3847e3bbe8b3436c52e445f7caf2a5bfba86a5fe"
DATA_DIR = "lk_fisheries"

def git(*args):
    subprocess.run(["git", "-C", DATA_DIR, *args], check=True, capture_output=True, text=True)

if not os.path.exists(os.path.join(DATA_DIR, ".git")):
    os.makedirs(DATA_DIR, exist_ok=True)
    git("init", "-q")
    git("remote", "add", "origin", REPO)
    git("fetch", "-q", "--depth", "1", "origin", SHA)
    git("checkout", "-q", SHA)

report_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "doc.xlsx"), recursive=True))
print(len(report_files), "weekly Excel reports downloaded")
print("example:", report_files[0])

## 2. Combine the weekly reports into one dataset
Each Excel report has a **Wholesale** sheet (Peliyagoda market) and a **Retail/Consumer** sheet (Colombo markets),
each listing about 30 species with: item no. | Sinhala name | English name | price last year | price last week | **price this week**.

Item numbers and English spellings change between years, but the Sinhala name stays the same, so species are matched
by Sinhala name. '-' or blank cells mean no price was recorded that week.

In [ ]:
def to_number(x):
    try:
        return float(str(x).replace(",", ""))
    except ValueError:
        return None                     

def read_report(path):
    """Read one weekly Excel report and return its species rows from both sheets."""
    folder = os.path.basename(os.path.dirname(path))
    date = folder[:10]
    rows = []
    for sheet_name, sheet in pd.read_excel(path, sheet_name=None, header=None, dtype=str).items():
        if sheet.empty:
            continue
        heading = " ".join(sheet.head(3).fillna("").astype(str).values.ravel()).lower() + " " + sheet_name.lower()
        if "wholesale" in heading:
            market = "wholesale"
        elif "consumer" in heading or "retail" in heading:
            market = "retail"
        else:
            continue                     # notes / empty helper sheets
        for _, r in sheet.iterrows():
            if not re.fullmatch(r"\d{1,2}", str(r[0]).strip()):
                continue                
            rows.append(dict(date=date, market=market,
                             sinhala=re.sub(r"\s+", "", str(r[1])), species=str(r[2]).strip(),
                             price_last_year=to_number(r[3]), price_last_week=to_number(r[4]),
                             price=to_number(r[5])))
    return rows

all_rows = []
for f in report_files:
    all_rows.extend(read_report(f))
raw = pd.DataFrame(all_rows)
raw["date"] = pd.to_datetime(raw["date"])

# The Sinhala name identifies the species; use its most common English spelling as the label.
english = raw.groupby("sinhala")["species"].agg(lambda s: s.value_counts().index[0])
raw["species"] = raw["sinhala"].map(english)
raw = raw.drop_duplicates(["date", "market", "sinhala"])

raw.to_csv("results/fish_prices_long.csv", index=False)
print(raw.shape, "| reports:", raw["date"].nunique(), "| species:", raw["sinhala"].nunique())
print(raw.groupby("market").size())
raw.head()

## 3. Clean the data

In [ ]:
df = raw.copy()

# 'Squids /Cuttle fish' appears in only 2 reports; it is a duplicate label of 'Cuttle fish'
df = df[df["species"] != "Squids /Cuttle fish"]

# A price of 0 Rs/kg means "not available that week", not free fish -> treat as missing
price_cols = ["price", "price_last_year", "price_last_week"]
df[price_cols] = df[price_cols].where(df[price_cols] > 0)

# Tidy species names (some contain quotes, double spaces or missing spaces)
def tidy(name):
    name = name.replace('"', "").replace("  ", " ")
    name = re.sub(r"\s*\(", " (", name).strip()
    return re.sub(r"\s+3$", "", name)          # 'Prawns (M) 3' -> 'Prawns (M)'
df["species"] = df["species"].map(tidy)

# One row per (species, date) with a wholesale and a retail column
wide = (df.pivot_table(index=["species", "date"], columns="market", values="price")
          .reset_index().sort_values(["species", "date"]).reset_index(drop=True))
wide.columns.name = None
print(wide.shape, "| species:", wide["species"].nunique(),
      "| weeks:", wide["date"].nunique())
print("Missing share:\n", wide[["retail", "wholesale"]].isna().mean().round(3))

## 4. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
med = wide.groupby("date")[["wholesale", "retail"]].median()
med.plot(ax=ax[0], marker=".", lw=1)
ax[0].set_title("Median price across 31 species (Rs/kg)")
ax[0].axvspan(pd.Timestamp("2022-04-01"), pd.Timestamp("2022-12-31"), color="red", alpha=0.1, label="2022 crisis")
ax[0].legend()

markup = (wide["retail"] / wide["wholesale"]).dropna()
markup.clip(0.5, 3).hist(bins=60, ax=ax[1])
ax[1].set_title(f"Retail / wholesale markup (median = {markup.median():.2f}x)")
plt.tight_layout(); plt.show()

print("Reports per year:")
print(wide.groupby(wide["date"].dt.year)["date"].nunique())

**EDA summary:**
- Prices roughly doubled between 2021 and 2023 (the economic crisis and fuel costs).
- 2020-2021 has only about 22 reports (COVID disruption), so the series has gaps.
- Retail is typically about 1.5x wholesale (middle half: 1.3-1.7x), varying by species and week.

## 5. Feature engineering
**Target:** the retail price this week.
**Key design choice:** the models predict the **log markup**, `log(retail / wholesale)`, not the raw retail price,
then convert back with `retail = wholesale × exp(prediction)`. This keeps predictions in a stable range even as
overall price levels rise over time (tree models can't extrapolate beyond their training range). Section 10.5 tests this.

| Feature | Meaning |
|---|---|
| `log_wholesale` | this week's wholesale price (log) |
| `log_wholesale_lag1` | last report's wholesale price |
| `log_wholesale_roll4` | 4-report average wholesale price |
| `wholesale_change` | % change in wholesale since last report (log difference) |
| `log_market_index` | median wholesale price of all species this week (general price level / inflation) |
| `month`, `week` | season |
| `sw_monsoon`, `ne_monsoon` | monsoon seasons that affect fishing (SW: May-Sep, NE: Dec-Feb) |
| `sp_*` | species (one-hot encoded) |

In [ ]:
def make_features(w):
    w = w.sort_values(["species", "date"]).copy()
    g = w.groupby("species")["wholesale"]
    w["wholesale_lag1"] = g.shift(1)
    w["wholesale_roll4"] = g.transform(lambda s: s.rolling(4, min_periods=1).mean())
    w["market_index"] = w.groupby("date")["wholesale"].transform("median")
    out = pd.DataFrame(index=w.index)
    out["log_wholesale"] = np.log(w["wholesale"])
    out["log_wholesale_lag1"] = np.log(w["wholesale_lag1"])
    out["log_wholesale_roll4"] = np.log(w["wholesale_roll4"])
    out["wholesale_change"] = out["log_wholesale"] - out["log_wholesale_lag1"]
    out["log_market_index"] = np.log(w["market_index"])
    out["month"] = w["date"].dt.month
    out["week"] = w["date"].dt.isocalendar().week.astype(int)
    out["sw_monsoon"] = out["month"].isin([5, 6, 7, 8, 9]).astype(int)
    out["ne_monsoon"] = out["month"].isin([12, 1, 2]).astype(int)
    sp = pd.get_dummies(w["species"], prefix="sp", dtype=float)
    sp.columns = [re.sub(r"\W+", "_", c) for c in sp.columns]
    return pd.concat([out, sp], axis=1), w

# The wholesale price must be known to predict retail; rows without a retail price have no label.
X_all, W = make_features(wide)
mask = W["wholesale"].notna() & W["retail"].notna()
X_all, W = X_all[mask], W[mask]
# First report of each species has no lag -> fill with this week's value (no change)
X_all["log_wholesale_lag1"] = X_all["log_wholesale_lag1"].fillna(X_all["log_wholesale"])
X_all["wholesale_change"] = X_all["wholesale_change"].fillna(0)

y_all = np.log(W["retail"] / W["wholesale"])   # target = log markup
FEATURES = list(X_all.columns)
print(X_all.shape, "features:", len(FEATURES))

**Note:** instead of predicting the retail price directly, I predict the markup (retail ÷ wholesale) and convert it
back afterwards. I did this because the later years (2024-2026) have higher prices than anything the model saw
during training, and tree-based models struggle to predict outside the range they were trained on. The markup
stays in a narrower, more stable range, so it's an easier target.

## 6. Train/test split (by time, not random)
- **Train:** Jan 2019 - Dec 2023
- **Test:** Jan 2024 - Apr 2026 (the most recent and more expensive period)

A random split would leak information across time, so the split is chronological. Inside the training set,
**TimeSeriesSplit (5 folds)** is used for cross-validation, so each fold only trains on the past.

In [ ]:
order = np.argsort(W["date"].values, kind="stable")
X_all, y_all, W = X_all.iloc[order], y_all.iloc[order], W.iloc[order]

is_train = (W["date"] < "2024-01-01").values
X_tr, X_te = X_all[is_train], X_all[~is_train]
y_tr, y_te = y_all[is_train], y_all[~is_train]
W_tr, W_te = W[is_train], W[~is_train]
print(f"train {len(X_tr)} rows ({W_tr.date.min().date()} → {W_tr.date.max().date()})")
print(f"test  {len(X_te)} rows ({W_te.date.min().date()} → {W_te.date.max().date()})")

def to_price(log_markup, wholesale):
    return wholesale * np.exp(log_markup)

def scores(actual, pred):
    return dict(MAE=mean_absolute_error(actual, pred),
                RMSE=np.sqrt(mean_squared_error(actual, pred)),
                MAPE=100 * mean_absolute_percentage_error(actual, pred),
                R2=r2_score(actual, pred))

**Note:** I split train/test by date instead of randomly, since a random split would let the model see prices from
the same week in both train and test, which would make it look better than it actually is. I'm using MAPE as the
main score because it's a percentage, so it's easy to compare across cheap and expensive species.

## 7. Models
| Family | Model | Idea |
|---|---|---|
| Baseline | **Species median markup** | retail = wholesale × that species' typical markup from training |
| Baseline | Ridge regression | linear model |
| Single tree | Decision tree | high-variance learner; this is what bagging fixes |
| **Bagging** | Bagging (decision trees) | 200 trees on bootstrap samples, averaged |
| **Bagging** | Random forest | bagging + random feature subsets at each split |
| **Boosting** | AdaBoost | re-weights samples the previous trees got wrong |
| **Boosting** | Gradient boosting (GBM) | each tree fits the residuals of the ensemble so far |
| **Boosting** | LightGBM | fast histogram-based gradient boosting |
| **Boosting** | CatBoost | gradient boosting with ordered boosting and symmetric trees |
| **Stacking** | Stacking | out-of-fold predictions of RF, Bagging, LightGBM, CatBoost and Ridge combined by a Ridge meta-model |
| **Hybrid** | RF + ANN | average of a random forest and a neural network (MLP), one of the lecturer's suggested hybrids |

In [ ]:
def build_models():
    return {
        "Ridge": Ridge(alpha=1.0),
        "Decision Tree": DecisionTreeRegressor(random_state=SEED),
        "Bagging (DT)": BaggingRegressor(DecisionTreeRegressor(), n_estimators=200,
                                         random_state=SEED, n_jobs=-1),
        "Random Forest": RandomForestRegressor(n_estimators=400, max_features=0.5, min_samples_leaf=2,
                                               random_state=SEED, n_jobs=-1),
        "AdaBoost": AdaBoostRegressor(DecisionTreeRegressor(max_depth=4), n_estimators=200,
                                      learning_rate=0.05, random_state=SEED),
        "GBM": GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=3,
                                         subsample=0.8, random_state=SEED),
        "LightGBM": LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.8,
                                  subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1),
        "CatBoost": CatBoostRegressor(iterations=800, learning_rate=0.05, depth=6,
                                      random_seed=SEED, verbose=0),
        "Stacking": StackingRegressor(
            estimators=[
                ("rf", RandomForestRegressor(n_estimators=300, max_features=0.5, min_samples_leaf=2,
                                             random_state=SEED, n_jobs=-1)),
                ("bag", BaggingRegressor(DecisionTreeRegressor(), n_estimators=100,
                                         random_state=SEED, n_jobs=-1)),
                ("lgbm", LGBMRegressor(n_estimators=600, learning_rate=0.03, subsample=0.8, subsample_freq=1,
                                       colsample_bytree=0.8, random_state=SEED, verbose=-1)),
                ("cat", CatBoostRegressor(iterations=800, learning_rate=0.05, depth=6,
                                          random_seed=SEED, verbose=0)),
                ("ridge", Ridge(alpha=1.0)),
            ],
            final_estimator=RidgeCV(alphas=[0.01, 0.1, 1, 10]),
            cv=KFold(5, shuffle=False), n_jobs=1),
        "Hybrid RF + ANN": VotingRegressor([
            ("rf", RandomForestRegressor(n_estimators=400, max_features=0.5, min_samples_leaf=2,
                                         random_state=SEED, n_jobs=-1)),
            ("ann", make_pipeline(StandardScaler(),
                                  MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=2000,
                                               early_stopping=True, random_state=SEED))),
        ]),
    }
FAMILY = {"Species median (baseline)": "Baseline", "Ridge": "Baseline", "Decision Tree": "Single tree",
          "Bagging (DT)": "Bagging", "Random Forest": "Bagging", "AdaBoost": "Boosting", "GBM": "Boosting",
          "LightGBM": "Boosting", "CatBoost": "Boosting", "Stacking": "Stacking", "Hybrid RF + ANN": "Hybrid"}

**Note:** I didn't run a full hyperparameter search for every model, mostly picked common/sensible values (e.g.
200-400 trees for the bagging and forest models, a smaller learning rate for boosting so it doesn't overfit).
Section 10.2 later shows that adding more trees past a point doesn't really help, which backs this up.

## 8. Time-series cross-validation on the training period (mean ± std)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []
for name, model in build_models().items():
    t0 = time.time(); fold_mape = []
    for tr_idx, va_idx in tscv.split(X_tr):
        model.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx])
        p = to_price(model.predict(X_tr.iloc[va_idx]), W_tr["wholesale"].values[va_idx])
        fold_mape.append(100 * mean_absolute_percentage_error(W_tr["retail"].values[va_idx], p))
    cv_rows.append(dict(model=name, family=FAMILY[name], cv_mape_mean=np.mean(fold_mape),
                        cv_mape_std=np.std(fold_mape), seconds=time.time() - t0))
    print(f"{name:18s} CV MAPE {np.mean(fold_mape):5.2f}% ± {np.std(fold_mape):4.2f}")
cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv("results/metrics_cv.csv", index=False)

## 9. Final training on 2019–2023 and testing on 2024–2026

In [ ]:
# Baseline: each species' median markup in the training period
species_cols = [c for c in FEATURES if c.startswith("sp_")]
train_species = W_tr["species"].values
median_markup = pd.Series(y_tr.values).groupby(train_species).median()
global_markup = float(np.median(y_tr))
def baseline_predict(Wpart):
    return Wpart["species"].map(median_markup).fillna(global_markup).values

fitted, test_rows, train_rows = {}, [], []
preds = pd.DataFrame({"date": W_te["date"].values, "species": W_te["species"].values,
                      "wholesale": W_te["wholesale"].values, "actual": W_te["retail"].values})

p_base = to_price(baseline_predict(W_te), W_te["wholesale"].values)
preds["Species median (baseline)"] = p_base
test_rows.append(dict(model="Species median (baseline)", **scores(W_te["retail"], p_base)))
train_rows.append(dict(model="Species median (baseline)",
                       **scores(W_tr["retail"], to_price(baseline_predict(W_tr), W_tr["wholesale"].values))))

for name, model in build_models().items():
    t0 = time.time()
    model.fit(X_tr, y_tr)
    fitted[name] = model
    p_te = to_price(model.predict(X_te), W_te["wholesale"].values)
    p_tr = to_price(model.predict(X_tr), W_tr["wholesale"].values)
    preds[name] = p_te
    test_rows.append(dict(model=name, **scores(W_te["retail"], p_te), fit_seconds=time.time() - t0))
    train_rows.append(dict(model=name, **scores(W_tr["retail"], p_tr)))
    print(f"{name:18s} test MAPE {test_rows[-1]['MAPE']:5.2f}%  ({time.time()-t0:.1f}s)")

test_df = pd.DataFrame(test_rows); test_df["family"] = test_df["model"].map(FAMILY)
train_df = pd.DataFrame(train_rows)
leader = (test_df.merge(train_df[["model", "MAPE"]].rename(columns={"MAPE": "train_MAPE"}), on="model")
                 .merge(cv_df[["model", "cv_mape_mean", "cv_mape_std"]], on="model", how="left")
                 .sort_values("MAPE"))
leader.to_csv("results/metrics_test.csv", index=False)
preds.to_csv("results/predictions_test.csv", index=False)
leader[["model", "family", "MAE", "RMSE", "MAPE", "R2", "train_MAPE", "cv_mape_mean", "cv_mape_std"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {"Baseline": "grey", "Single tree": "black", "Bagging": "tab:green", "Boosting": "tab:orange",
          "Stacking": "tab:purple", "Hybrid": "tab:blue"}
lb = leader.sort_values("MAPE", ascending=False)
ax.barh(lb["model"], lb["MAPE"], color=lb["family"].map(colors))
ax.set_xlabel("Test MAPE (%) – lower is better"); ax.set_title("Model comparison on 2024–2026 test data")
plt.tight_layout(); plt.show()

## 10. Why do the models behave this way?

### 10.1 Bias vs variance: training error vs test error

In [ ]:
bv = leader[["model", "family", "train_MAPE", "MAPE"]].rename(columns={"MAPE": "test_MAPE"})
bv["gap"] = bv["test_MAPE"] - bv["train_MAPE"]
bv.to_csv("results/bias_variance.csv", index=False)
bv.round(2)

### 10.2 Effect of the number of trees: random forest (bagging) vs GBM (boosting)

In [ ]:
curve = []
for n in [1, 5, 10, 25, 50, 100, 200, 400]:
    rf = RandomForestRegressor(n_estimators=n, max_features=0.5, min_samples_leaf=2,
                               random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
    curve.append(dict(model="Random Forest", n_trees=n,
                      train_MAPE=100*mean_absolute_percentage_error(W_tr.retail, to_price(rf.predict(X_tr), W_tr.wholesale)),
                      test_MAPE=100*mean_absolute_percentage_error(W_te.retail, to_price(rf.predict(X_te), W_te.wholesale))))
gbm = GradientBoostingRegressor(n_estimators=1500, learning_rate=0.05, max_depth=3, subsample=0.8,
                                random_state=SEED).fit(X_tr, y_tr)
tr_stage = list(gbm.staged_predict(X_tr)); te_stage = list(gbm.staged_predict(X_te))
for n in [1, 5, 10, 25, 50, 100, 200, 400, 800, 1500]:
    curve.append(dict(model="GBM", n_trees=n,
                      train_MAPE=100*mean_absolute_percentage_error(W_tr.retail, to_price(tr_stage[n-1], W_tr.wholesale)),
                      test_MAPE=100*mean_absolute_percentage_error(W_te.retail, to_price(te_stage[n-1], W_te.wholesale))))
curve = pd.DataFrame(curve); curve.to_csv("results/ntrees_curve.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, m in zip(ax, ["Random Forest", "GBM"]):
    c = curve[curve.model == m]
    a.plot(c.n_trees, c.train_MAPE, "o-", label="train"); a.plot(c.n_trees, c.test_MAPE, "o-", label="test")
    a.set_xscale("log"); a.set_title(m); a.set_xlabel("number of trees"); a.legend()
ax[0].set_ylabel("MAPE (%)"); plt.tight_layout(); plt.show()
curve.round(2)

### 10.3 Why stacking helps only a little: base-model errors are correlated

In [ ]:
base_names = ["Random Forest", "Bagging (DT)", "LightGBM", "CatBoost", "Ridge", "Hybrid RF + ANN"]
resid = pd.DataFrame({m: preds[m] - preds["actual"] for m in base_names})
corr = resid.corr(); corr.to_csv("results/residual_corr.csv")
stack = fitted["Stacking"]
weights = pd.DataFrame({"base_model": ["Random Forest", "Bagging (DT)", "LightGBM", "CatBoost", "Ridge"],
                        "meta_weight": stack.final_estimator_.coef_})
weights.to_csv("results/stack_weights.csv", index=False)
print(corr.round(2)); print(); print(weights.round(3))

### 10.4 Which features matter? (random forest importance, species dummies summed)

In [ ]:
imp = pd.Series(fitted["Random Forest"].feature_importances_, index=FEATURES)
grouped = imp[[c for c in FEATURES if not c.startswith("sp_")]].copy()
grouped["species (all one-hot columns)"] = imp[[c for c in FEATURES if c.startswith("sp_")]].sum()
grouped = grouped.sort_values(ascending=False)
grouped.rename("importance").to_frame().to_csv("results/importance.csv")
grouped.plot.barh(figsize=(8, 4), title="Random forest feature importance"); plt.gca().invert_yaxis(); plt.show()

### 10.5 Raw price target vs markup target (the extrapolation problem)

In [ ]:
lvl_rows = []
y_tr_level, y_te_level = np.log(W_tr["retail"]), np.log(W_te["retail"])
for name in ["Ridge", "Random Forest", "LightGBM"]:
    m = build_models()[name].fit(X_tr, y_tr_level)
    p = np.exp(m.predict(X_te))
    lvl_rows.append(dict(model=name, target="raw log price", MAPE=100*mean_absolute_percentage_error(W_te.retail, p)))
    lvl_rows.append(dict(model=name, target="log markup", MAPE=float(leader.set_index("model").loc[name, "MAPE"])))
    if name == "Random Forest": preds["Random Forest (raw-price target)"] = p
lvl = pd.DataFrame(lvl_rows); lvl.to_csv("results/level_vs_markup.csv", index=False)
print(lvl.pivot(index="model", columns="target", values="MAPE").round(2))
print("Highest retail price in training:", W_tr.retail.max(), "| in test:", W_te.retail.max())
preds.to_csv("results/predictions_test.csv", index=False)

### 10.6 Where does the best model make mistakes? (error by species)

In [ ]:
best = leader.iloc[0]["model"]; print("Best model on test:", best)
sp_err = (preds.assign(err_best=lambda d: (d[best] - d.actual).abs() / d.actual * 100,
                       err_base=lambda d: (d["Species median (baseline)"] - d.actual).abs() / d.actual * 100)
               .groupby("species")[["err_best", "err_base"]].mean().sort_values("err_best"))
sp_err.columns = [f"{best} MAPE %", "Baseline MAPE %"]
sp_err.to_csv("results/species_error.csv")
sp_err.round(1)

## 11. Save models for the dashboard
The full random forest, bagging and stacking models are too large for a free Streamlit Cloud app and for GitHub,
so the dashboard uses lighter refits with fewer trees (section 10.2 shows the forest gains almost nothing after
about 100 trees). Their test scores are saved too, so the dashboard reports honest numbers.

In [ ]:
def light_models():
    rf100 = lambda: RandomForestRegressor(n_estimators=100, max_features=0.5, min_samples_leaf=3,
                                          random_state=SEED, n_jobs=-1)
    bag50 = lambda: BaggingRegressor(DecisionTreeRegressor(min_samples_leaf=2), n_estimators=50,
                                     random_state=SEED, n_jobs=-1)
    full = build_models()
    return {
        "Random Forest": rf100(), "Bagging (DT)": bag50(),
        "LightGBM": full["LightGBM"], "CatBoost": full["CatBoost"], "Ridge": full["Ridge"],
        "Stacking": StackingRegressor(
            estimators=[("rf", rf100()), ("bag", bag50()),
                        ("lgbm", full["LightGBM"]), ("cat", full["CatBoost"]), ("ridge", Ridge(alpha=1.0))],
            final_estimator=RidgeCV(alphas=[0.01, 0.1, 1, 10]), cv=KFold(5, shuffle=False)),
        "Hybrid RF + ANN": VotingRegressor([("rf", rf100()), ("ann", make_pipeline(StandardScaler(),
                              MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=2000,
                                           early_stopping=True, random_state=SEED)))]),
    }
app_models, app_rows = {}, []
for name, m in light_models().items():
    m.fit(X_tr, y_tr); app_models[name] = m
    p = to_price(m.predict(X_te), W_te["wholesale"].values)
    app_rows.append(dict(model=name, **scores(W_te["retail"], p)))
pd.DataFrame(app_rows).to_csv("results/app_models_test.csv", index=False)
print(pd.DataFrame(app_rows).round(2))

In [ ]:
latest = W.sort_values("date").groupby("species").tail(1).set_index("species")
bundle = dict(
    models=app_models,
    features=FEATURES, median_markup=median_markup.to_dict(), global_markup=global_markup,
    species=sorted(W["species"].unique()),
    latest_wholesale=latest["wholesale"].to_dict(),
    latest_market_index=float(np.exp(X_all["log_market_index"].iloc[-1])),
    best_model=best, train_end="2023-12-31", test_start="2024-01-01",
)
joblib.dump(bundle, "models/model_bundle.joblib", compress=3)
wide.to_csv("results/wide_prices.csv", index=False)
print("Saved:", os.path.getsize("models/model_bundle.joblib") / 1e6, "MB")

## 12. References
- Data: Ministry of Fisheries weekly fish price reports, via `github.com/nuuuwan/lk_datasets` (dataset 024).
- scikit-learn, LightGBM and CatBoost documentation.